# Day 4: Data Cleaning

## Finding Duplicates
Duplicate rows — the same record entered or imported twice — inflate counts, distort averages, and can make a model overfit to repeated examples, so checking for them is one of the first cleaning steps on any new dataset. `df.duplicated()` returns a `True`/`False` flag for each row, marking a row `True` only when an earlier identical row already exists, while `df.drop_duplicates()` returns a new DataFrame with those repeat rows removed (keeping the first occurrence by default). In the output, row 2 (`Alice`, 25) is flagged `True` because it's an exact repeat of row 0, and after `drop_duplicates()` only the unique rows remain.

In [ ]:
import pandas as pd

df = pd.DataFrame({'Name': ['Alice', 'Bob', 'Alice'], 'Age': [25, 30, 25]})
print(df.duplicated())       # True marks rows that repeat an earlier row exactly
print(df.drop_duplicates())  # returns the DataFrame with those repeats removed

## Fixing Data Types
When data is loaded from a CSV or scraped from a web source, numeric-looking columns often end up stored as text (strings), which silently breaks any math or comparison you try to do on them — so verifying and fixing `dtypes` is a routine cleaning step. `df.dtypes` lists the data type pandas is using for each column, and `.astype(int)` converts a column's values from one type to another, here from text to integer. The output shows the `Price` column's type changing from `str` (object) before the conversion to `int64` after, meaning the values can now be used in numeric operations like sums or comparisons.

In [ ]:
df2 = pd.DataFrame({'Price': ['100', '200', '300']})
print(df2.dtypes)  # Price is stored as text (str/object), not a number
df2['Price'] = df2['Price'].astype(int)  # convert the column's values to integers
print(df2.dtypes)  # now Price is int64 and can be used in numeric calculations

## Cleaning Text Columns
Text data entered by different people (or systems) is rarely consistent — extra whitespace, inconsistent capitalization, or abbreviations — and these small inconsistencies cause "New York" and " new york " to be treated as different categories unless you standardize them first. Pandas' `.str` accessor exposes string methods (like Python's built-in string methods) that work across an entire column at once: `.str.strip()` removes leading/trailing whitespace, and `.str.title()` capitalizes the first letter of each word. The output shows the messy `City` values cleaned up and consistently formatted — `' New York '` becomes `'New York'`, `'la'` becomes `'La'`, and `'CHICAGO'` becomes `'Chicago'`.

In [ ]:
df3 = pd.DataFrame({'City': [' New York ', 'la', 'CHICAGO']})
# strip() removes leading/trailing spaces, title() standardizes capitalization
df3['City'] = df3['City'].str.strip().str.title()
print(df3)

## Renaming Columns
Column names imported from databases or external systems are often cryptic abbreviations (`emp_nm`, `sal`), which makes code harder to read and more error-prone to work with — renaming them to clear, consistent names early on saves confusion later. `.rename(columns={...})` takes a dictionary mapping old names to new names and returns a DataFrame with those columns relabeled, leaving the data itself untouched. The output shows the same data as before, just with `emp_nm` now labeled `Name` and `sal` now labeled `Salary`.

In [ ]:
df4 = pd.DataFrame({'emp_nm': ['Alice'], 'sal': [50000]})
# rename() maps old column names to new ones without touching the underlying data
df4 = df4.rename(columns={'emp_nm': 'Name', 'sal': 'Salary'})
print(df4)

## Detecting Outliers (IQR Method)
An outlier is a value that's unusually far from the rest of the data (a typo, a sensor glitch, or a genuinely rare event), and it can badly distort statistics like the mean or a regression fit if left unaddressed — so outlier detection is a standard part of cleaning. The IQR (interquartile range) method defines "normal" as the middle 50% of the data (between the 25th and 75th percentiles) and flags anything more than 1.5x that range below Q1 or above Q3 as an outlier — a widely used, robust rule of thumb. In the example, most values cluster around 10-13, but 90 is far outside the calculated `lower`/`upper` bounds, so it's the only value that shows up in the `outliers` result.

In [ ]:
data = pd.Series([10, 12, 11, 13, 12, 90, 11, 10])
Q1, Q3 = data.quantile(0.25), data.quantile(0.75)  # 25th and 75th percentile values
IQR = Q3 - Q1  # interquartile range: spread of the "normal" middle 50% of the data
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR  # standard 1.5*IQR outlier fences
outliers = data[(data < lower) | (data > upper)]  # keep only values outside the fences
print(outliers)